# The Display List

The **display list** is an internal record of all the graphical objects
that have been drawn on the current page. It enables grid_py to
replay, edit, and inspect the scene after the fact -- without
requiring the user to keep track of every grob manually.

This notebook covers:

* Enabling / disabling display-list recording.
* Replaying the display list with `grid_refresh`.
* Applying a function to every entry with `grid_dl_apply`.
* Recording callable expressions with `grid_record`.

The narrative follows the original R vignette (`displaylist.Rnw`).

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grid_py as gp

## Recording

By default, every drawing operation (e.g., `grid_rect`, `grid_text`)
is appended to the display list. You can toggle recording on and off
with `grid_display_list(on)`.

In [ ]:
# Start a new page (display list recording is on by default)
gp.grid_newpage()

gp.grid_rect(gp=gp.Gpar(fill="lightyellow"), name="bg")
gp.grid_circle(
    r=gp.Unit(0.2, "npc"),
    gp=gp.Gpar(fill="steelblue"),
    name="dot",
)
gp.grid_text("recorded", y=gp.Unit(0.15, "npc"), name="label")

In [ ]:
# Check what is on the display list
gp.grid_ls()

## Disabling and Re-enabling the Display List

`grid_display_list(False)` disables recording. Grobs drawn while
recording is off will not be captured, so they cannot be replayed
or edited later.

In [ ]:
# Turn off recording
prev = gp.grid_display_list(False)
print("Previous recording state:", prev)

# This rectangle will NOT be on the display list
gp.grid_rect(
    x=gp.Unit(0.8, "npc"), y=gp.Unit(0.8, "npc"),
    width=gp.Unit(0.15, "npc"), height=gp.Unit(0.15, "npc"),
    gp=gp.Gpar(fill="red"),
    name="unrecorded",
)

# Turn recording back on
gp.grid_display_list(True)
print("Display list contents after off/on cycle:")
gp.grid_ls()

## Replaying: grid_refresh

`grid_refresh()` replays all entries on the display list, effectively
redrawing the entire scene. This is useful after editing the display
list or resizing the figure.

In [ ]:
# Replay the display list
gp.grid_refresh()

## grid_dl_apply

`grid_dl_apply(fn)` applies a function to every item on the display
list, replacing each entry with the function's return value. This is
a powerful (and dangerous!) way to post-process an entire scene.

In [ ]:
gp.grid_newpage()

gp.grid_rect(gp=gp.Gpar(fill="lightgrey"), name="bg")
gp.grid_circle(
    r=gp.Unit(0.15, "npc"),
    gp=gp.Gpar(fill="cornflowerblue"),
    name="c1",
)

# Define a function that changes the gp of every grob
def make_bold(item):
    """Set lwd=3 on every grob that has a gp."""
    if hasattr(item, '_gp') and item._gp is not None:
        item._gp = gp.Gpar(lwd=3)
    return item

gp.grid_dl_apply(make_bold)
gp.grid_refresh()

## grid_record

`grid_record(expr)` wraps a callable in a special grob. When the
display list is replayed, the callable is re-executed, which means
the output can adapt to changes in viewport size or state.

In [ ]:
gp.grid_newpage()

def draw_adaptive():
    """Draw a circle whose label shows the current viewport path."""
    gp.grid_circle(
        r=gp.Unit(0.3, "npc"),
        gp=gp.Gpar(fill="peachpuff"),
    )
    gp.grid_text("adaptive")

gp.grid_record(draw_adaptive, name="adaptive_grob")

## Summary

* The display list automatically records drawing operations.
* `grid_display_list(on)` toggles recording.
* `grid_refresh()` replays the display list to redraw the scene.
* `grid_dl_apply(fn)` transforms every entry on the display list.
* `grid_record(expr)` captures a callable for deferred / adaptive drawing.